In [ ]:
import yfinance as yf
from pandas import Timestamp
from tqdm.notebook import tqdm
import pandas as pd
import os
import numpy as np
import seaborn as sns 
from tqdm import tqdm
import matplotlib.pyplot as plt

def segment(timestamp,seg):
    # return str((timestamp.year,(timestamp.week-1)))
    w = timestamp.week-1
    y=timestamp.year
    m=timestamp.month-1

    if w == 52:
        # y += 1
        w = 0
    if seg=="Weekly":
        return f"({y}, {w})"
    if seg=="Monthly":
        return f"({y}, {m})"
# def getTickers(seg,tickers):
#     series=None
#     for ticker in tqdm(tickers):
#         tk = yf.Ticker(ticker)
#         if seg=="Weekly":
#             prices=tk.history(period="6y",interval="1wk")
#             prices["snapshot"]=[segment(Timestamp(t),seg) for t in prices.index.values]
#         if "Monthly" in seg:
#             prices=tk.history(period="6y",interval="1mo")
#             prices["snapshot"]=[segment(Timestamp(t),seg) for t in prices.index.values]
#         if seg=="Trimester":
#             prices=tk.history(period="6y",interval="3mo")
#             prices["snapshot"]=[segment(Timestamp(t),seg) for t in prices.index.values]

#         prices.index=prices["snapshot"]
#         prices[ticker]=prices["Close"]
#         if series is None:
#             series=prices[[ticker]]
#         else:
#             series=series.join(prices[ticker])
#     series=series.groupby(series.index).first()    
#     return series.drop_duplicates()

In [ ]:
def metadataFilter(NODES,EDGES=None):
    new_nodes=NODES.copy()
    # new_nodes=new_nodes[new_nodes["volume"]<new_nodes["volume"].quantile(.9)]
    if EDGES is not None:
        new_edges=EDGES.copy()
        # nodes_ids=set(new_nodes.index)
        # new_edges=new_edges[new_edges["source"].isin(nodes_ids)]
        # new_edges=new_edges[new_edges["target"].isin(nodes_ids)]
        return new_nodes,new_edges
    return new_nodes


In [ ]:
def getTickers(seg):
    btc=pd.read_csv("../input/BTC-USD.csv",index_col=0)
    if seg=="Weekly":
        btc["snapshot"]=[segment(Timestamp(t),seg) for t in btc.index.values]
    if seg=="Monthly":
        btc["snapshot"]=[segment(Timestamp(t),seg) for t in btc.index.values]
    btc.index=btc.snapshot
    
    btc=btc[["Close"]]
    btc["BTC-USD"]=btc.Close
    btc=btc[["BTC-USD"]]
    btc=btc.loc[sorted(btc.index,key=lambda x: eval(x)[0]*52+eval(x)[1])]
    btc=btc.groupby(btc.index).first()
    return btc


# Loading Weekly Data

In [ ]:
path=f"../graphsV2/NFTLevel-Pruning_V2-Weekly2"
seg="Weekly"
SOGLIA=.5
SOGLIA_CORRELAZIONE=.0

tickers=["BTC-USD"]
prices=getTickers(seg)
tickers=list(prices.columns.values)

w_snaps = sorted(os.listdir(path), key=lambda x: eval(x)[0]*100+eval(x)[1])


x=[]
LENS=[]
for snap in tqdm(w_snaps):
    df=pd.read_csv(f"{path}/{snap}/metadata.csv")
    df=metadataFilter(df)
    LENS.append((len(df),len(df.drop_duplicates(keep="first"))))
    x.append({
        "snapshot":snap,
        "sum(volume)":df["volume"].sum(),
        "mean(volume)":df["volume"].mean(),

        "mean(max(price))":df["max_price"].mean(),
        "mean(min(price))":df["min_price"].mean(),
        "mean(mean(price))":df["avg_price"].mean(),
        
        "sum(#tx)":df["#tx"].sum(),
        "max(#tx)":df["#tx"].max(),
        "min(#tx)":df["#tx"].min(),
        "mean(#tx)":df["#tx"].mean(),
        })
volumes=pd.DataFrame.from_records(x)
volumes.index=volumes.snapshot

assert sum([l[0]-l[1] for l in LENS])==0

import math

def loadSnap(snap):
        try:
                nodes=pd.read_csv(f"{path}/{snap}/metadata.csv",index_col=0)
                edges=pd.read_csv(f"{path}/{snap}/edge_list.csv").drop_duplicates(keep="first")
                nodes,edges=metadataFilter(nodes,edges)
                edges=edges[edges.weight>SOGLIA]

                edges["category_src"]=edges["source"].apply(lambda x: nodes.loc[x]["category"])
                edges["category_dst"]=edges["target"].apply(lambda x: nodes.loc[x]["category"])
                intra=edges[edges.category_src==edges.category_dst].weight.mean()
                if math.isnan(intra):
                        intra=0
                inter=edges[edges.category_src!=edges.category_dst].weight.mean()
                if math.isnan(inter):
                        inter=0
                in_degs={node:len(d) for node,d in edges.groupby("target")}
                out_degs={node:len(d) for node,d in edges.groupby("source")}
                return {
                        "snapshot":snap,
                        "sum(sim)":edges["weight"].sum(),
                        "mean(sim)":edges["weight"].mean(),
                        "intra":intra,
                        "inter":inter,
                        "mean(in_deg)":np.mean(list(in_degs.values())),
                        "mean(out_deg)":np.mean(list(out_degs.values())),

                        # "mean(in_deg)":np.mean([in_degs[n] if n in in_degs else 0 for n in nodes.index.values]),
                        # "mean(out_deg)":np.mean([out_degs[n] if n in in_degs else 0 for n in nodes.index.values]),
                }

        except Exception as e:
                return {
                        "snapshot":snap,
                        "sum(sim)":0,
                        "mean(sim)":0,
                        "intra":0,
                        "inter":0,
                        "mean(in_deg)":0,
                        "mean(out_deg)":0,
                }    
from p_tqdm import p_umap

sim = p_umap(loadSnap, w_snaps)

sim=pd.DataFrame().from_records(sim)
sim.index=sim.snapshot

nodes=pd.read_csv(f"{path}/{w_snaps[-1]}/metadata.csv",index_col=0)
nodes=metadataFilter(nodes)
BORN_NFT={snap:0 for snap in w_snaps}
BORN_COLL={snap:0 for snap in w_snaps}
#per ogni nodo mi ero salvato il tempo della prima transazione come tempo di snapshot
for snap,nfts_born in nodes.groupby("first_seen_snap"):
    BORN_NFT[snap]=len(nfts_born)

for coll,nfts in nodes.groupby("collection"):
    snap_born_coll=nfts["first_seen_snap"][nfts["first_seen_snap"].apply(lambda x: eval(x)[0]*100+eval(x)[1]).argmin()]
    try:
        BORN_COLL[snap_born_coll]=BORN_COLL[snap_born_coll]+1
    except:
        pass
sum(list(BORN_NFT.values())),sum(list(BORN_COLL.values()))


keys=set(prices.index.values)
keys.intersection_update(volumes.index.values)

keys.intersection_update(sim.index.values)

keys=list(set(keys))

keys=sorted(keys,key=lambda x: eval(x)[0]*52+eval(x)[1])
BORN_COLL={k:BORN_COLL[k] for k in keys}
BORN_NFT={k:BORN_NFT[k] for k in keys}
BORN_NFT=[{"snapshot":k,"born_nft":BORN_NFT[k]} for k in BORN_NFT]
BORN_COLL=[{"snapshot":k,"born_coll":BORN_COLL[k]} for k in BORN_COLL]
BORN_NFT=pd.DataFrame().from_records(BORN_NFT)
BORN_COLL=pd.DataFrame().from_records(BORN_COLL)
BORN_COLL.index=BORN_COLL["snapshot"]
BORN_NFT.index=BORN_NFT["snapshot"]

df=pd.concat([volumes.loc[keys],sim.loc[keys],prices.loc[keys],BORN_NFT,BORN_COLL],axis=1)[list(volumes.columns.values)[1:]+list(sim.columns.values)[1:]+tickers+["born_nft","born_coll"]]
df=df.loc[keys]


WEEKLY={
        "snaps":w_snaps,
        "stats":x,
        "sims":sim,
        "BORN_COLL":BORN_COLL,
        "BORN_NFT":BORN_NFT,
        "time_series":df
}



# Loading Monthly Data

In [ ]:
path=f"../graphsV2/NFTLevel-Pruning_V2"
seg="Monthly"
SOGLIA=.5
SOGLIA_CORRELAZIONE=.0


tickers=["BTC-USD"]
prices=getTickers(seg)
tickers=list(prices.columns.values)

w_snaps = sorted(os.listdir(path), key=lambda x: eval(x)[0]*100+eval(x)[1])



x=[]
LENS=[]
for snap in tqdm(w_snaps):
    df=pd.read_csv(f"{path}/{snap}/metadata.csv")
    df=metadataFilter(df)
    LENS.append((len(df),len(df.drop_duplicates(keep="first"))))
    x.append({
        "snapshot":snap,
        "sum(volume)":df["volume"].sum(),
        "mean(volume)":df["volume"].mean(),

        "mean(max(price))":df["max_price"].mean(),
        "mean(min(price))":df["min_price"].mean(),
        "mean(mean(price))":df["avg_price"].mean(),
        
        "sum(#tx)":df["#tx"].sum(),
        "max(#tx)":df["#tx"].max(),
        "min(#tx)":df["#tx"].min(),
        "mean(#tx)":df["#tx"].mean(),
        })
volumes=pd.DataFrame.from_records(x)
volumes.index=volumes.snapshot

assert sum([l[0]-l[1] for l in LENS])==0

import math

def loadSnap(snap):

        try:
                nodes=pd.read_csv(f"{path}/{snap}/metadata.csv",index_col=0)
                edges=pd.read_csv(f"{path}/{snap}/edge_list.csv").drop_duplicates(keep="first")
                nodes,edges=metadataFilter(nodes,edges)
                edges=edges[edges.weight>SOGLIA]
                edges["category_src"]=edges["source"].apply(lambda x: nodes.loc[x]["category"])
                edges["category_dst"]=edges["target"].apply(lambda x: nodes.loc[x]["category"])
                intra=edges[edges.category_src==edges.category_dst].weight.mean()
                if math.isnan(intra):
                        intra=0
                inter=edges[edges.category_src!=edges.category_dst].weight.mean()
                if math.isnan(inter):
                        inter=0
                in_degs={node:len(d)    for node,d in edges.groupby("target")}
                out_degs={node:len(d) for node,d in edges.groupby("source")}
                return {
                        "snapshot":snap,
                        "sum(sim)":edges["weight"].sum(),
                        "mean(sim)":edges["weight"].mean(),
                        "intra":intra,
                        "inter":inter,
                        "mean(in_deg)":np.mean(list(in_degs.values())),
                        "mean(out_deg)":np.mean(list(out_degs.values())),
                        # "mean(in_deg)":np.mean([in_degs[n] if n in in_degs else 0 for n in nodes.index.values]),
                        # "mean(out_deg)":np.mean([out_degs[n] if n in in_degs else 0 for n in nodes.index.values]),
                }

        except Exception as e:
                return {
                        "snapshot":snap,
                        "sum(sim)":0,
                        "mean(sim)":0,
                        "intra":0,
                        "inter":0,
                        "mean(in_deg)":0,
                        "mean(out_deg)":0,
                }   
from p_tqdm import p_umap

sim = p_umap(loadSnap, w_snaps)

sim=pd.DataFrame().from_records(sim)
sim.index=sim.snapshot

nodes=pd.read_csv(f"{path}/{w_snaps[-1]}/metadata.csv",index_col=0)
nodes=metadataFilter(nodes)

BORN_NFT={snap:0 for snap in w_snaps}
BORN_COLL={snap:0 for snap in w_snaps}
#per ogni nodo mi ero salvato il tempo della prima transazione come tempo di snapshot
for snap,nfts_born in nodes.groupby("first_seen_snap"):
    BORN_NFT[snap]=len(nfts_born)

for coll,nfts in nodes.groupby("collection"):
    snap_born_coll=nfts["first_seen_snap"][nfts["first_seen_snap"].apply(lambda x: eval(x)[0]*100+eval(x)[1]).argmin()]
    try:
        BORN_COLL[snap_born_coll]=BORN_COLL[snap_born_coll]+1
    except:
        pass
sum(list(BORN_NFT.values())),sum(list(BORN_COLL.values()))


keys=set(prices.index.values)
keys.intersection_update(volumes.index.values)

keys.intersection_update(sim.index.values)

keys=list(set(keys))

keys=sorted(keys,key=lambda x: eval(x)[0]*52+eval(x)[1])
BORN_COLL={k:BORN_COLL[k] for k in keys}
BORN_NFT={k:BORN_NFT[k] for k in keys}
BORN_NFT=[{"snapshot":k,"born_nft":BORN_NFT[k]} for k in BORN_NFT]
BORN_COLL=[{"snapshot":k,"born_coll":BORN_COLL[k]} for k in BORN_COLL]
BORN_NFT=pd.DataFrame().from_records(BORN_NFT)
BORN_COLL=pd.DataFrame().from_records(BORN_COLL)
BORN_COLL.index=BORN_COLL["snapshot"]
BORN_NFT.index=BORN_NFT["snapshot"]
df=pd.concat([volumes.loc[keys],sim.loc[keys],prices.loc[keys],BORN_NFT,BORN_COLL],axis=1)[list(volumes.columns.values)[1:]+list(sim.columns.values)[1:]+tickers+["born_nft","born_coll"]]
df=df.loc[keys]


MONTHLY={
        "snaps":w_snaps,
        "stats":x,
        "sims":sim,
        "BORN_COLL":BORN_COLL,
        "BORN_NFT":BORN_NFT,
        "time_series":df
}

In [ ]:
print("SETTIMANALE")
WEEKLY["time_series"]

In [ ]:
import matplotlib.pyplot as plt
BORN_NFT=MONTHLY["BORN_NFT"]["born_nft"]
BORN_COLL=MONTHLY["BORN_COLL"]["born_coll"]

plt.figure(figsize=(20,10))
plt.title("\#NFT nati per snapshot")
born=BORN_NFT
plt.plot(list(BORN_NFT.keys()),born/born.sum(),label="\% Nascite")
alive=np.cumsum(born)
plt.plot(list(BORN_NFT.keys()),alive/alive.max(),label="\% Alive")
plt.plot(list(BORN_NFT.keys()),df["intra"].values/df["inter"].values,label="intra/inter")
plt.plot(list(BORN_NFT.keys()),df["intra"].values,label="intra")
plt.plot(list(BORN_NFT.keys()),df["inter"].values,label="inter")
plt.plot(list(BORN_NFT.keys()),[0 for _ in df["inter"].values],linestyle="dashed")
plt.plot(list(BORN_NFT.keys()),[1 for _ in df["inter"].values],linestyle="dashed")
plt.legend()
plt.xticks(rotation=90)
plt.show()

import matplotlib.pyplot as plt
plt.figure(figsize=(20,10))
plt.title("\#Collection nate per snapshot")
born=BORN_COLL
plt.plot(list(BORN_COLL.keys()),born/born.sum(),label="\% Nascite")
alive=np.cumsum(born)
plt.plot(list(BORN_COLL.keys()),alive/alive.max(),label="\% Alive")
plt.plot(list(BORN_COLL.keys()),df["intra"].values/df["inter"].values,label="intra/inter")
plt.plot(list(BORN_COLL.keys()),df["intra"].values,label="intra")
plt.plot(list(BORN_COLL.keys()),df["inter"].values,label="inter")
plt.plot(list(BORN_COLL.keys()),[0 for _ in df["inter"].values],linestyle="dashed")
plt.plot(list(BORN_COLL.keys()),[1 for _ in df["inter"].values],linestyle="dashed")
plt.legend()
plt.xticks(rotation=90)
plt.show()

In [ ]:
MONTHLY["time_series"].describe()

In [ ]:
WEEKLY["time_series"].describe()

In [ ]:
def L2_loss(y_1,y_2):
    return ((y_1-y_2)**2).mean()
def crosscorr(datax, datay, lag=0, wrap=False):
    if lag<0:
        q1=datax[:lag]
        q2=datay[-lag:]
    elif lag==0:
        q1=datax
        q2=datay
    else:
        q1=datax[lag:]
        q2=datay[:-lag]
    q1=np.nan_to_num(q1)
    q2=np.nan_to_num(q2)
    return np.corrcoef(q1,q2)[0,1],q1,q2
def crossL2(datax, datay, lag=0, wrap=False):
    """ Lag-N cross correlation. 
    Shifted data filled with NaNs 
    
    Parameters
    ----------
    lag : int, default 0
    datax, datay : pandas.Series objects of equal length
    Returns
    ----------
    crosscorr : float
    """
    datax=datax.copy().reset_index()
    datay=datay.copy().reset_index()
    
    shiftedy = datay.shift(lag)
    shiftedy.iloc[:lag] = datay.iloc[-lag:].values


    return L2_loss(datax,datay.shift(lag)) 


In [ ]:
df.columns.values

In [ ]:

Q1={}
Q2={}
CROSS_CORR={}
WEEKLY["TLCC"]={
    "Q1":Q1,
    "Q2":Q2,
    "CROSS_CORR":CROSS_CORR
}
MULTIPLIER=4




df=WEEKLY["time_series"]
for lag in range(-13*MULTIPLIER,13*MULTIPLIER):
    curr_cross_corr={}
    q1_={}
    q2_={}
    for q1 in df.columns.values:
        
        for q2 in df.columns.values:
            # c2,c1=c1,c2
            if q1 not in curr_cross_corr:
                curr_cross_corr[q1]={}
            q_1=df[q1].values
            q_2=df[q2].values

            # q_2=np.where(np.abs(q_2)<1e-2,1,q_2)            
            corr,q_1,q_2=crosscorr(q_1, q_2, lag=lag)
            if np.isnan(q_1).any():
                print(q1)
            if np.isnan(q_2).any():
                print(q2)
            # corr = crossL2(df[c1], df[c2], lag=lag)
            curr_cross_corr[q1][q2]=corr
            q1_[q1]=q_1
            q2_[q2]=q_2
        CROSS_CORR[lag]=pd.DataFrame.from_dict(curr_cross_corr)        
        Q1[lag]=q1_      
        Q2[lag]=q2_

In [ ]:

Q1={}
Q2={}
CROSS_CORR={}
MONTHLY["TLCC"]={
    "Q1":Q1,
    "Q2":Q2,
    "CROSS_CORR":CROSS_CORR
}
MULTIPLIER=1

df=MONTHLY["time_series"]
for lag in range(-12*MULTIPLIER,12*MULTIPLIER):
    curr_cross_corr={}
    q1_={}
    q2_={}
    for q1 in df.columns.values:
        
        for q2 in df.columns.values:
            if q1 not in curr_cross_corr:
                curr_cross_corr[q1]={}
            q_1=df[q1].values
            q_2=df[q2].values
            corr,q_1,q_2=crosscorr(q_1, q_2, lag=lag)
            curr_cross_corr[q1][q2]=corr
            q1_[q1]=q_1
            q2_[q2]=q_2
        CROSS_CORR[lag]=pd.DataFrame.from_dict(curr_cross_corr)        
        Q1[lag]=q1_      
        Q2[lag]=q2_

In [ ]:
l=6*4
q2="mean(sim)"
q1="mean(mean(price))"
Q1=WEEKLY["TLCC"]["Q1"]
Q2=WEEKLY["TLCC"]["Q2"]
df=WEEKLY["time_series"]

fig,ax=plt.subplots(1,1,figsize=(30,10))
fig.suptitle(f"Lag={l}")
if l>0:
    ax.plot(np.array(list(range(len(Q1[l][q1]))))+l,Q1[l][q1],label=f"q1={q1}",linewidth=5,color="red")
else:
    ax.plot(np.array(list(range(len(Q1[l][q1])))),Q1[l][q1],color="red",label=f"q1={q1}",linewidth=5)
ax.plot(df[q1].values,color="red",linestyle=":",label=f"q1_original={q1}")
ax.legend(loc=4)
ax.set_yscale("log")

ax=ax.twinx()
if l<0:
    ax.plot(np.array(list(range(-l,len(df[q2])))),Q2[l][q2],label=f"q2={q2}",linewidth=5)
else:
    ax.plot(Q2[l][q2],label=f"q2={q2}",linewidth=5)
ax.plot(df[q2].values,color="blue",linestyle=":",label=f"q2_original={q2}")
ax.set_xticklabels(ax.get_xticks(), rotation = 0)
ax.legend(loc=0)
plt.savefig(f"../graphsV2/plots/NFT-Level-TLCC-Lag {l} {seg}.png")
plt.show()

In [ ]:
l=6*1
q2="mean(sim)"
q1="mean(mean(price))"
Q1=MONTHLY["TLCC"]["Q1"]
Q2=MONTHLY["TLCC"]["Q2"]
df=MONTHLY["time_series"]

fig,ax=plt.subplots(1,1,figsize=(30,10))
fig.suptitle(f"Lag={l}")
if l>0:
    ax.plot(np.array(list(range(len(Q1[l][q1]))))+l,Q1[l][q1],label=f"q1={q1}",linewidth=5,color="red")
else:
    ax.plot(np.array(list(range(len(Q1[l][q1])))),Q1[l][q1],color="red",label=f"q1={q1}",linewidth=5)
ax.plot(df[q1].values,color="red",linestyle=":",label=f"q1_original={q1}")
ax.legend(loc=4)
ax.set_yscale("log")

ax=ax.twinx()
if l<0:
    ax.plot(np.array(list(range(-l,len(df[q2])))),Q2[l][q2],label=f"q2={q2}",linewidth=5)
else:
    ax.plot(Q2[l][q2],label=f"q2={q2}",linewidth=5)
ax.plot(df[q2].values,color="blue",linestyle=":",label=f"q2_original={q2}")
ax.set_xticklabels(ax.get_xticks(), rotation = 0)
ax.legend(loc=0)
plt.savefig(f"../graphsV2/plots/NFT-Level-TLCC-Lag {l} {seg}.png")
plt.show()

Con offset negativo il leader è la grandezza a sinistra, mentre il follower quella a destra,
altrimenti viceversa.

In [ ]:
print("OFFSET mensili considerati")
MONTHLY["TLCC"]["Q2"].keys()

In [ ]:
print("OFFSET settimanali considerati")
WEEKLY["TLCC"]["Q2"].keys()

In [ ]:
print("Massima lunghezza temporale della serie mensile ",len(MONTHLY["TLCC"]["Q2"][0]["mean(sim)"]))
print("Minima lunghezza temporale della serie mensile ",len(MONTHLY["TLCC"]["Q2"][-11]["mean(sim)"]))

In [ ]:
print("Massima lunghezza temporale della serie settimanale ",len(WEEKLY["TLCC"]["Q2"][0]["mean(sim)"]))
print("Minima lunghezza temporale della serie settimanale ",len(WEEKLY["TLCC"]["Q2"][51]["mean(sim)"]))

In [ ]:
import matplotlib
font = {'family' : 'normal',
        'weight' : 'bold',
        'size'   : 50}
OVERRIDE_BEST={}

matplotlib.rc('font', **font)
plt.rcParams['text.usetex'] = True
CROSS_CORR_M=MONTHLY["TLCC"]["CROSS_CORR"]
CROSS_CORR_W=WEEKLY["TLCC"]["CROSS_CORR"]

In [ ]:
serie1=MONTHLY["time_series"]["mean(sim)"]
serie2=WEEKLY["time_series"]["mean(sim)"]

fig,ax=plt.subplots(1,1,figsize=(21,12),sharex=True)
ax2=ax.twiny()

ax.set_xlabel(f"Months")
ax2.set_xlabel(f"Weeks")

ax.plot(serie1.values,color="red", linestyle="--",linewidth=4,label=f"Monthly", dashes=(8, 4),alpha=0.6)
ax2.plot(serie2.values,color="blue", linestyle=":",linewidth=4,label=f"Weekly")

ax2.legend(loc="upper left")
ax.legend(loc="lower right")

ax.set_ylabel("Average similarity")
ax.set_ylim(0.51,0.6)
plt.grid()
plt.savefig("../graphsV2/plots/Average Similarity.png")
plt.show()

In [ ]:
inter_m=MONTHLY["time_series"]["inter"]
intra_m=MONTHLY["time_series"]["intra"]
mean_m=MONTHLY["time_series"]["mean(sim)"]

inter_w=WEEKLY["time_series"]["inter"]
intra_w=WEEKLY["time_series"]["intra"]
mean_w=WEEKLY["time_series"]["mean(sim)"]

fig,axs=plt.subplots(2,1,figsize=(21,24))
ax=axs[0]
ax.set_title("Monthly")
ax.plot(inter_m.values,color="red", linestyle=":",linewidth=4,label=f"Inter")
ax.plot(intra_m.values,color="blue", linestyle=":",linewidth=4,label=f"Intra")
ax.plot(mean_m.values,color="green", linestyle="--",linewidth=4,label=f"Average", dashes=(8, 4),alpha=0.6)
ax.grid()
ax.set_ylim(0.51,0.6)
ax.legend(loc="lower right")
ax.set_ylabel("Similarities")

ax=axs[1]
ax.set_title("Weekly")
ax.plot(inter_w.values,color="red", linestyle=":",linewidth=4,label=f"Inter")
ax.plot(intra_w.values,color="blue", linestyle=":",linewidth=4,label=f"Intra")
ax.plot(mean_w.values,color="green", linestyle="--",linewidth=4,label=f"Average", dashes=(8, 4),alpha=0.6)
ax.grid()
ax.set_ylim(0.51,0.6)
ax.legend(loc="lower right")
ax.set_ylabel("Similarities")

plt.savefig("../graphsV2/plots/All-Sim.png")
plt.show()

In [ ]:
serie1=MONTHLY["time_series"]["mean(mean(price))"]
serie2=WEEKLY["time_series"]["mean(mean(price))"]

fig,ax=plt.subplots(1,1,figsize=(21,12),sharex=True)
ax2=ax.twiny()

ax.set_xlabel(f"Months")
ax2.set_xlabel(f"Weeks")

ax.plot(serie1.values,color="red", linestyle="--",linewidth=4,label=f"Monthly", dashes=(8, 4),alpha=0.6)
ax2.plot(serie2.values,color="blue", linestyle=":",linewidth=4,label=f"Weekly")

ax2.legend(loc='upper right')
ax.legend(loc="center right",bbox_to_anchor=(1, .75), )

ax.set_ylabel("Average NFT selling price (\$)")
plt.grid()
plt.savefig("../graphsV2/plots/Average NFT selling price.png")
plt.show()

In [ ]:
df=MONTHLY["time_series"].copy()
df["idx"]=list(range(len(df)))
df[["idx","BTC-USD"]].iloc[-1]

In [ ]:
serie1=MONTHLY["time_series"]["BTC-USD"]
serie2=WEEKLY["time_series"]["BTC-USD"]

fig,ax=plt.subplots(1,1,figsize=(21,12),sharex=True)
ax2=ax.twiny()

ax.set_xlabel(f"Months")
ax2.set_xlabel(f"Weeks")

ax.plot(serie1.values,color="red", linestyle="--",linewidth=4,label=f"Monthly", dashes=(8, 4),alpha=0.6)
ax2.plot(serie2.values,color="blue", linestyle=":",linewidth=4,label=f"Weekly")

ax2.legend(loc="upper left")
ax.legend(loc="center left",bbox_to_anchor=(0.0, .75), )

ax.set_ylabel("BTC-USD market close price (\$)")
plt.grid()
plt.savefig("../graphsV2/plots/BTC-USD market close price.png")
plt.show()

In [ ]:
serie1=MONTHLY["time_series"]["born_nft"]
serie2=WEEKLY["time_series"]["born_nft"]

fig,ax=plt.subplots(1,1,figsize=(21,12),sharex=True)
ax2=ax.twiny()

ax.set_xlabel(f"Months")
ax2.set_xlabel(f"Weeks")

ax.plot(serie1.values,color="red", linestyle="--",linewidth=4,label=f"Monthly", dashes=(8, 4),alpha=0.6)
ax2.plot(serie2.values,color="blue", linestyle=":",linewidth=4,label=f"Weekly")

ax2.legend(loc="upper left")
ax.legend(loc="center left",bbox_to_anchor=(0.0, .75), )

ax.set_ylabel("\# First sold NFTs")
plt.grid()
plt.savefig("../graphsV2/plots/Born NFT.png")
plt.show()

In [ ]:
serie1=MONTHLY["time_series"]["born_coll"]
serie2=WEEKLY["time_series"]["born_coll"]

fig,ax=plt.subplots(1,1,figsize=(21,12),sharex=True)
ax2=ax.twiny()

ax.set_xlabel(f"Months")
ax2.set_xlabel(f"Weeks")

ax.plot(serie1.values,color="red", linestyle="--",linewidth=4,label=f"Monthly", dashes=(8, 4),alpha=0.6)
ax2.plot(serie2.values,color="blue", linestyle=":",linewidth=4,label=f"Weekly")

ax2.legend(loc="upper left")
ax.legend(loc="center left",bbox_to_anchor=(0.0, .75), )

ax.set_ylabel("\# Born Collections")
plt.grid()
plt.savefig("../graphsV2/plots/Born COLL.png")
plt.show()

In [ ]:
def plot_custom(q1,q2,path_plots="../graphsV2/plots"):
    fig,ax=plt.subplots(1,1,figsize=(21,12),sharex=True)
    ax2=ax.twiny()
    ax.set_xlim([(-51-5)/4,(51+5)/4])
    ax2.set_xlim([-51-5,51+5])

    ax.set_ylim([-1,1])
    ax2.set_ylim([-1,1])
    
    lags_M=list(CROSS_CORR_M.keys())
    lags_W=list(CROSS_CORR_W.keys())  

    datas_W=[CROSS_CORR_W[lag].loc[q2][q1] for lag in lags_W]
    datas_M=[CROSS_CORR_M[lag].loc[q2][q1]  for lag in lags_M]
    t_synch=np.argmax(np.abs(datas_M))
    ax.plot(lags_M, datas_M,color="red", linestyle="--",linewidth=4,label=f"Monthly", dashes=(8, 4),alpha=0.6)
    ax.scatter(lags_M[t_synch],datas_M[t_synch],color='r',s=600)
    ax.legend(loc="lower right")
    ax.set_xlabel(f"Offset (months)")
        
    t_synch=np.argmax(np.abs(datas_W))
    ax2.plot(lags_W, datas_W,color="blue", linestyle=":",linewidth=4,label=f"Weekly")
    ax2.scatter(lags_W[t_synch],datas_W[t_synch],color='b',s=600)
    ax2.set_xlabel(f"Offset (weeks)")
    ax2.axvline(0,-1,1,linestyle="--",color="gray")

    # ax.set_xlim([np.min(datas_W),np.max(datas_W)])
    ax2.legend(loc="upper left")
    ax.set_ylabel("Correlation")
    ax2.grid(None)

    
    os.makedirs(path_plots,exist_ok=True)
    plt.savefig(f"{path_plots}/NFT-Level-TLCC-q1={q1} q2={q2}.png")
    plt.show()

In [ ]:
q1="mean(mean(price))"#mean(mean(price))
q2="mean(sim)"#BTC-USD,mean(sim)
plot_custom(q1,q2)
print(q1,"VS",q2)

In [ ]:

q1="mean(mean(price))"#mean(mean(price))
q2="BTC-USD"#BTC-USD,mean(sim)
plot_custom(q1,q2)
print(q1,"VS",q2)

In [ ]:

q1="mean(mean(price))"#mean(mean(price))
q2="born_coll"#BTC-USD,mean(sim),born_coll,born_nft
plot_custom(q1,q2)
print(q1,"VS",q2)

In [ ]:
q1="mean(mean(price))"#mean(mean(price))
q2="born_nft"#BTC-USD,mean(sim),born_coll,born_nft
plot_custom(q1,q2)
print(q1,"VS",q2)

In [ ]:
df

In [ ]:
for q1 in ['max(#tx)','min(#tx)', 'mean(#tx)', 
       'mean(in_deg)', 'mean(out_deg)', 
        'BTC-USD']:
    for q2 in [ 'mean(volume)', 
        'mean(max(price))', 'mean(mean(price))', 
        'mean(sim)', 'intra', 'inter',
       'born_nft','born_coll']:
        plot_custom(q1,q2,path_plots="../graphsV2/plots/all_plots")
        print(q1,"VS",q2)        

In [ ]:
q1="mean(#tx)"#mean(mean(price))
q2="inter"#BTC-USD,mean(sim),born_coll,born_nft
plot_custom(q1,q2)
print(q1,"VS",q2)

In [ ]:
q1="mean(#tx)"#mean(mean(price))
q2="intra"#BTC-USD,mean(sim),born_coll,born_nft
plot_custom(q1,q2)
print(q1,"VS",q2)